In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SILVER_FILES = "/data/silver/*.parquet"
QUARANTINE_FILES = "/data/quarantine/*.parquet"

spark = SparkSession.builder.master("spark://spark-master:7077").appName("exploratory").getOrCreate()

In [4]:
silver_df = spark.read.parquet(SILVER_FILES)
silver_df.printSchema()
print(silver_df.count())
silver_df.show(5)

quarantine_df = spark.read.parquet(QUARANTINE_FILES)
quarantine_df.show(5)

root
 |-- rental_id: string (nullable = true)
 |-- bike_id: string (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _load_date: date (nullable = true)
 |-- _file_name: string (nullable = true)

6664571
+---------+-------+-------------------+-------------------+--------+----------------+--------------------+--------------+--------------------+--------------------+----------+--------------------+
|rental_id|bike_id|         start_date|           end_date|duration|start_station_id|  start_station_name|end_station_id|    end_station_name|_ingestion_timestamp|_load_date|          _file_name|
+---------+-------+-------------------+-----------------

In [16]:
quarantine_df_agg = (
    quarantine_df
        .groupBy("rejection_reason")
        .agg(
            F.count("rejection_reason").alias("total_rows_impacted"),
            F.countDistinct("_file_name").alias("unique_files_affected"),
            F.countDistinct("Rental Id").alias("unique_rental_id")
        )
)

quarantine_df_agg.show()

+-------------------+-------------------+---------------------+----------------+
|   rejection_reason|total_rows_impacted|unique_files_affected|unique_rental_id|
+-------------------+-------------------+---------------------+----------------+
|   missing_end_date|              11898|                    2|           11898|
|missing_end_station|                172|                   10|             156|
|   invalid_duration|               7555|                   10|            6825|
+-------------------+-------------------+---------------------+----------------+



In [5]:
spark.stop()